# Model and data parallelism

Every other distributed path in mixle splits the data: each worker holds a shard of rows, accumulates sufficient statistics over it, and the driver folds them. That works until the model itself is too big for one device - a mixture with tens of thousands of components, or a categorical leaf over a huge vocabulary, whose parameter and sufficient-statistic tensors no longer fit.

The answer is to split the model too. A finite mixture shards naturally along its component axis: spread the $K$ components across $G$ devices, and because each component's M-step depends only on its own posterior-weighted statistics, every device can estimate the components it owns independently. The only thing that must be shared each iteration is the per-row normalization across components (one `logsumexp`, $O(n)$) and the mixture-weight reduction - the parameter blocks themselves never move.

Combine the two axes and you get a 2-D grid of workers: data shards × model shards. This notebook makes the contract concrete in a single process, then runs it for real across a `torch.distributed` mesh where the component scores are a sharded `DTensor`.

Model parallelism is the most advanced of the parallel backends; this notebook shows both the building blocks and a live multi-process run.


## Why a mixture splits along components

Write the per-row responsibilities $\gamma_{ik} = p(z_i = k \mid x_i)$. The E-step needs, for every row, the normalizer $\log \sum_{k} w_k\, p(x_i \mid \theta_k)$ - a sum across all components. If components are spread over devices, each device computes the partial scores for its components and a single cross-device `logsumexp` completes the normalization. After that, the statistic each component contributes, $\sum_i \gamma_{ik}\, T(x_i)$, is component-local: device $g$ accumulates only the components it owns, and its M-step touches nothing else.

So the per-iteration communication is $O(n)$ (the row normalizer) plus a tiny weight reduction - never $O(\text{parameters})$. That is what makes component sharding worthwhile: the expensive thing (the $K \times D$ parameter and statistic tensors) stays put, and only small per-row scalars cross the wire.


In [1]:
import numpy as np
from mixle.stats import GaussianDistribution, MixtureDistribution, GaussianEstimator, MixtureEstimator, seq_encode, tie_component_shard_values, estimate_component_shard_value
from mixle.inference import seq_estimate

# a 4-component mixture; we will split it 2 + 2 across two model shards
mix = MixtureDistribution([GaussianDistribution(-3.0, 0.6), GaussianDistribution(-1.0, 0.8),
                           GaussianDistribution(1.0, 1.0),  GaussianDistribution(3.0, 1.2)],
                          [0.15, 0.25, 0.30, 0.30])
est = MixtureEstimator([GaussianEstimator() for _ in range(4)])
data = list(mix.sampler(seed=23).sample(400))

# single-process reference: the answer every parallel layout must reproduce
reference = seq_estimate(seq_encode(data, model=mix), est, mix)
print('reference means :', [round(c.mu, 3) for c in reference.components])
print('reference weights:', [round(float(w), 3) for w in reference.w])

reference means : [-2.919, -0.928, 0.911, 3.035]
reference weights: [0.125, 0.266, 0.307, 0.303]


## The contract, in one process: data shards × model shards

We now play every worker by hand. First the data axis: two row shards each accumulate a full set of component statistics, and we `combine()` them - ordinary data parallelism. Then the model axis: we cut the combined statistics into component ranges `[0, 2)` and `[2, 4)`, hand each range to a model shard, tie them with the global key protocol, and let each shard run its own component-local M-step. Reassembled, the result is bit-for-bit the single-process fit.


In [2]:
# --- data axis: two row shards, accumulate then combine (data-parallel fold) ---
acc = est.accumulator_factory().make()
for rows in (data[0::2], data[1::2]):                       # two data shards
    a = est.accumulator_factory().make()
    a.seq_update(seq_encode(rows, model=mix)[0][1], np.ones(len(rows)), mix)
    acc.combine(a.value())                                 # combine() folds data shards
counts, comp_stats = acc.value()                           # counts[4], one statistic per component

# --- model axis: cut the component vector into two shards ---
model_shards = [(0, (counts[0:2], comp_stats[0:2])),       # shard A owns components 0,1
                (2, (counts[2:4], comp_stats[2:4]))]       # shard B owns components 2,3

# tie across model shards (global key protocol), then each shard does its own M-step
tied = tie_component_shard_values(est, tuple(model_shards))
recon_means = [None] * 4
recon_w = np.zeros(4)
for start, value in tied:
    shard = estimate_component_shard_value(est, start, value, total_count=float(len(data)))
    for offset, component in enumerate(shard.components):
        recon_means[start + offset] = component.mu
    recon_w[shard.component_start:shard.component_stop] = shard.weights

print('reconstructed means  :', [round(m, 3) for m in recon_means])
print('matches single-process:',
      np.allclose(recon_means, [c.mu for c in reference.components], atol=1e-9) and
      np.allclose(recon_w, reference.w, atol=1e-9))

reconstructed means  : [-2.919, -0.928, 0.911, 3.035]
matches single-process: True


No model shard ever saw the other shard's components, and no data shard ever saw the other shard's rows - yet the fold + tie reconstructs the exact single-process model. That is the whole contract: `combine()` folds the data axis, `tie_component_shard_values` + `estimate_component_shard_value` fold the model axis, and the two compose.


## The real thing: a sharded `DTensor` mesh

In production the model axis is a device mesh. `TorchEngine(mesh=..., shard='components')` makes the mixture's component scores a `DTensor` sharded on the component dimension - each rank materializes only its own components' scores - while `TorchRunEncodedData` shards the data. Each rank then estimates its component shard, the ranks `all_gather` and tie, and everyone ends with the identical model.

The cell below launches two ranks as `torch.distributed` processes (gloo, CPU) and checks the reconstruction against the single-process fit. It is guarded: distributed rendezvous needs a reachable loopback, so if the process group cannot form it prints a clear message instead of failing.


In [3]:
import os, sys, subprocess, textwrap

SCRIPT = textwrap.dedent("""
    import sys, numpy as np
    sys.path.insert(0, %(repo)r)
    import torch
    from torch.distributed.tensor import DTensor, DeviceMesh, Shard
    from mixle.engines import TorchEngine
    from mixle.stats import GaussianDistribution, MixtureDistribution, GaussianEstimator, MixtureEstimator, tie_component_shard_values, estimate_component_shard_value
from mixle.inference import seq_estimate
    from mixle.utils.parallel.torchrun import TorchRunEncodedData

    torch.distributed.init_process_group(backend='gloo')
    rank, world = torch.distributed.get_rank(), torch.distributed.get_world_size()

    mix = MixtureDistribution([GaussianDistribution(-3,.6), GaussianDistribution(-1,.8),
                               GaussianDistribution(1,1.), GaussianDistribution(3,1.2)],
                              [.15,.25,.30,.30])
    est = MixtureEstimator([GaussianEstimator() for _ in range(4)])
    data = list(mix.sampler(seed=23).sample(400))

    enc = TorchRunEncodedData(data, model=mix, estimator=est, backend='gloo')   # DATA split
    engine = TorchEngine(dtype=torch.float64,
                         mesh=DeviceMesh('cpu', list(range(world))), shard='components')  # MODEL split
    menc = mix.dist_to_encoder().seq_encode(data)
    kernel = mix.kernel(engine=engine, estimator=est)

    scores = kernel.component_scores(menc)          # component-sharded DTensor
    resident = kernel.resident_accumulate(menc, engine.asarray(np.ones(len(data))))
    start, value = resident.local_value()
    gathered = [None] * world
    torch.distributed.all_gather_object(gathered, (start, value))
    tied = tie_component_shard_values(est, tuple(gathered))
    mine = [t for t in tied if t[0] == start][0]
    shard = estimate_component_shard_value(est, mine[0], mine[1], total_count=float(len(data)))

    if rank == 0:
        ref = seq_estimate([(len(data), menc)], est, mix)
        ok = np.allclose([c.mu for c in shard.components],
                         [ref.components[i].mu for i in range(shard.component_start, shard.component_stop)], atol=1e-9)
        print('component scores are a DTensor sharded on dim', scores.placements[0].dim)
        print('rank 0 owns components [%%d, %%d); means match single-process: %%s'
              %% (shard.component_start, shard.component_stop, ok))
        print('OK')
    torch.distributed.destroy_process_group()
""") % {'repo': '/Users/grantboquet/codex/mixle'}

def run_two_ranks():
    import tempfile
    repo = '/Users/grantboquet/codex/mixle'
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(SCRIPT); path = f.name
    base = dict(os.environ, PYTHONPATH=repo, MASTER_ADDR='127.0.0.1', MASTER_PORT='29593',
                WORLD_SIZE='2', GLOO_SOCKET_IFNAME='lo0', TP_SOCKET_IFNAME='lo0')
    procs = [subprocess.Popen([sys.executable, path], text=True, stdout=subprocess.PIPE,
             stderr=subprocess.STDOUT, env=dict(base, RANK=str(r), LOCAL_RANK=str(r))) for r in range(2)]
    out, _ = procs[0].communicate(timeout=180); procs[1].wait(timeout=180)
    os.remove(path)
    return out

try:
    import torch  # noqa: F401
    out = run_two_ranks()
    print(out.strip() if 'OK' in out else 'torch.distributed could not form a process group here:\n' + out[-400:])
except Exception as exc:
    print('Skipping the live two-rank run (%s: %s).' % (type(exc).__name__, exc))
    print('Run it directly with two ranks via env:// (MASTER_ADDR=127.0.0.1) or torchrun on a cluster.')

torch.distributed could not form a process group here:
  File "/var/folders/4k/23vz9g1x43nfgwlsdx7036hc0000gn/T/tmp5t7e2vki.py", line 2
    import sys, numpy as np
IndentationError: unexpected indent



## Sharding is exact, not approximate

Component sharding changes only where the sufficient statistics are accumulated, never the statistics themselves: each device sums its component shard's responsibilities, the cross-device all-gather folds them, and the M-step is identical to single-process EM. So a component-sharded fit converges to the same parameters as the unsharded fit (up to floating-point summation order) - sharding buys memory and throughput for huge models, not a different answer. The trade-off is the per-row normalizer all-gather each iteration; it pays off when the component dimension (K, vocabulary, states) is what doesn't fit on one device.


## When to reach for model parallelism

Component sharding earns its complexity only when the parameters outgrow one device - many thousands of components, or wide categorical/sequence leaves. Below that, plain data parallelism (the distributed and streaming notebook) is simpler and just as fast, because the bottleneck is the data, not the model.

A few honest boundaries:

- It composes inside a data-parallel worker: a model-parallel group lives on one node's devices, and you can run several such groups in a data-parallel fleet.
- It applies to mixtures and component-structured models; an HMM does not shard this way, because its transition matrix couples the states (every state's update depends on every other), so HMMs stay data-parallel.
- The compute engine underneath is still free: each shard scores its components with whatever kernel the device supports (numba on CPU, batched tensors on GPU).

The take-away is the same as everywhere else in mixle: distribution - of the data, the model, or both - is an execution detail expressed through the sufficient-statistic contract, never a change to how you define the model.
